In [ ]:
import xarray as xr
import matplotlib.pylab as plt
import pint_xarray
from pathlib import Path
import json
import pandas as pd

In [ ]:
delta_coder = xr.coders.CFTimedeltaCoder(decode_via_units=True)
time_coder = xr.coders.CFDatetimeCoder()
inv_vars = ['inverse.stress_balance.velocity_scale', 'inverse.stress_balance.length_scale', 'inverse.tikhonov.penalty_weight']


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt

INV_DIR = Path("/Users/andy/base/pism-terra/ismip7_test/output/inverse")

files = sorted(INV_DIR.glob("inv_*_g2400m*.nc"))
print(f"found {len(files)} files")          # <-- if this is 0, it's the path

inv_vars = ["inverse.state_func",
            "inverse.stress_balance.velocity_scale",
            "inverse.stress_balance.length_scale",
            "inverse.tikhonov.penalty_weight"]
dss = []
rows = []
for p in files:
    try:
        with xr.open_dataset(p) as ds:
            r = {k.split(".")[-1]: ds.pism_config.attrs[k] for k in inv_vars}
            w   = ds["vel_misfit_weight"].squeeze().values.astype(float)   # misfit area Ω
            res = ds["inv_residual"].squeeze().values                      # |u_model - u_obs|, m/yr
            r["M"] = float(np.sqrt(np.nansum(w * res**2) / np.nansum(w)))   # physical RMS misfit (m/yr)
            r["N"] = float(np.sqrt(ds.J_design.isel(inv_iter=-1)))         # model norm
            rows.append(r)
    except Exception:
        pass
df = pd.DataFrame(rows).sort_values(["state_func", "velocity_scale","length_scale","penalty_weight"]).reset_index(drop=True)

def corner(n, m):
    """Index of max Menger curvature on a (log N, log M) L-curve."""
    x, y = np.log10(n), np.log10(m); kbest, best = -np.inf, None
    for i in range(1, len(x)-1):
        a = np.hypot(x[i]-x[i-1], y[i]-y[i-1]); b = np.hypot(x[i+1]-x[i], y[i+1]-y[i])
        c = np.hypot(x[i+1]-x[i-1], y[i+1]-y[i-1])
        area = abs((x[i]-x[i-1])*(y[i+1]-y[i-1]) - (x[i+1]-x[i-1])*(y[i]-y[i-1]))
        k = 2*area/(a*b*c) if a*b*c > 0 else 0
        if k > kbest: kbest, best = k, i
    return best

fig_all, ax_all = plt.subplots(figsize=(8,6))

for (loss, vs, ls), g in df.groupby(["state_func", "velocity_scale", "length_scale"]):
    fig, ax = plt.subplots(figsize=(8,6))
    g = g.sort_values("penalty_weight").reset_index(drop=True)
    ax.plot(g["N"], g["M"], "o-", ms=4, label=f"vs={vs:g}, ls={ls:g}")
    ax_all.plot(g["N"], g["M"], "o-", ms=4, label=f"{loss} vs={vs:g}, ls={ls:g}")
    for _, row in g.iterrows():
        ax.annotate(f"{row.penalty_weight:g}", (row.N, row.M), fontsize=6, xytext=(3,3), textcoords="offset points")
        if len(g) >= 3 and (i := corner(g["N"].values, g["M"].values)) is not None:
            ax.plot(g["N"][i], g["M"][i], "k*", ms=15, zorder=5)
            ax_all.plot(g["N"][i], g["M"][i], "k*", ms=15, zorder=5)
        #print(f"vs={vs:g}, ls={ls:g}: corner penalty={g[i]:.1f} m/yr, N={g.N[i]:.3f})")
    
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_ylim(120, 200)
    ax.set_xlabel(r"model norm  $N=\sqrt{J_\mathrm{design}}$")
    ax.set_ylabel(r"data misfit  $M$ (m yr$^{-1}$)")
    ax.set_title("L-curve (labels = penalty $\\eta$; ★ = corner)")
    ax.legend(fontsize=8); ax.grid(True, which="both", alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"lcurve_loss_{loss}_ls_{ls}_vs_{vs}.png", dpi=300)

ax_all.set_xscale("log"); ax.set_yscale("log")
ax_all.set_ylim(120, 200)
ax_all.set_xlabel(r"model norm  $N=\sqrt{J_\mathrm{design}}$")
ax_all.set_ylabel(r"data misfit  $M$ (m yr$^{-1}$)")
ax_all.set_title("L-curve (labels = penalty $\\eta$; ★ = corner)")
ax_all.legend(fontsize=8); ax.grid(True, which="both", alpha=0.3)
fig_all.tight_layout()
fig_all.savefig("lcurve.png", dpi=300)

In [ ]:
!open lcurve_*.png

In [ ]:
m = misfits.inv_misfit.stack(z=["state_func", "velocity_scale", "length_scale", "penalty_weight"])
m = m.drop_vars(['z', 'state_func', 'velocity_scale', 'length_scale', 'penalty_weight']).assign_coords(z=[str(t) for t in m.z.values])          # MultiIndex -> plain string labels
m.plot.line(x="inv_iter", hue="z", add_legend=False)

In [ ]:
ds.inv_J_misfit

In [ ]:
res

In [ ]:
dss[0].inv_J_misfit

In [ ]:
            pmisfit = ds.inv_misfit.exand_dims({k: [v] for k, v in r.items()})


In [ ]:
2.4e-7 * 1e12 / 1000 / 50 / 50